# QLoRA Fine-Tuning v2: Qwen2.5-1.5B-Instruct for SEO & GEO Pipeline
**Project:** AI SEO & GEO Blog Builder (Module 1 — Research & Content Brief) — **Model v2**
**Hardware:** Google Colab Free Tier (1x NVIDIA T4 GPU - 15GB VRAM)

## Nâng cấp v2 so với v1
| Hạng mục | v1 | v2 |
|---|---|---|
| Tổng mẫu train | 1,948 | ~2,430 |
| Tỷ lệ tiếng Việt | 35% | 50% |
| Số niches | 14 | 34 |
| Intent rules trong prompt | ❌ | ✅ Explicit + examples |
| known_intent trong CB | 33% | 60% |

**Hướng dẫn:** Upload `train_v2.jsonl` và `test_v2.jsonl` vào `/content/` trước khi chạy.

## 1. Environment & GPU Check

In [ ]:
!nvidia-smi

In [ ]:
!pip uninstall -y torchaudio
!pip install -q -U transformers datasets trl peft bitsandbytes accelerate

## 2. Load 4-Bit Base Model & Tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False

# Ensure all non-quantized weights and buffers are strictly float16 (no bfloat16 on T4 GPU)
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
for buffer in model.buffers():
    if buffer.dtype == torch.bfloat16:
        buffer.data = buffer.data.to(torch.float16)

## 3. Configure LoRA (PEFT)
Target all linear projections: `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# Ensure all LoRA trainable weights are float32 so GradScaler works reliably on T4 GPU
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.print_trainable_parameters()

## 4. Load Dataset & Format ChatML
**Instruction:** Drag and drop `train.jsonl` and `test.jsonl` from your computer into the Files pane on the left (`/content/`).

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "train_v2.jsonl",
    "test": "test_v2.jsonl"
})

def apply_chat_template(sample):
    text = tokenizer.apply_chat_template(
        sample["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

formatted_dataset = dataset.map(apply_chat_template)
print("Sample formatted prompt:\n", formatted_dataset["train"][0]["text"][:450])

## 5. Training with SFTTrainer (3 Epochs)

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./outputs",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_length=1024,
    dataset_text_field="text",
    packing=False,
    fp16=False,
    bf16=False,
    report_to="none"
)

# Ensure all LoRA trainable parameters are float32 and NO bfloat16 exists on T4 GPU
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
    elif param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
for buffer in model.buffers():
    if buffer.dtype == torch.bfloat16:
        buffer.data = buffer.data.to(torch.float16)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["test"],
    processing_class=tokenizer,
    args=training_args
)

trainer.train()

## 6. Save Adapter & Download

In [ ]:
ADAPTER_DIR = "./qwen2.5_1.5b_qlora_adapter"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

!zip -r qwen2.5_1.5b_qlora_adapter.zip ./qwen2.5_1.5b_qlora_adapter

from google.colab import files
files.download("qwen2.5_1.5b_qlora_adapter.zip")
print("[SUCCESS] Adapter packaging complete. Download started!")

## 7. Direct Inference Test

In [ ]:
system_cb = """You are an expert SEO & GEO content strategist for Shopify e-commerce blogs.
Given a primary keyword and optional context, return ONLY a JSON object (no markdown, no explanation) in this exact schema:
{
  \"primary_keyword\": \"<string>\",
  \"secondary_keywords\": [\"<string>\", ...],
  \"dominant_intent\": \"Informational | Commercial | Transactional | Navigational\",
  \"funnel_stage\": \"TOFU | MOFU | BOFU\",
  \"recommended_content_type\": \"Informational | Buyer's guide | How-to | FAQ\",
  \"recommended_style\": \"Informative | Analytical | Instructive | Persuasive | Conversational\",
  \"recommended_tone\": \"Authoritative | Caring | Friendly | Informative | Formal | Conversational\",
  \"target_audience\": \"<string>\",
  \"key_talking_points\": [\"<string with E-E-A-T: stat or credential>\", ...],
  \"geo_blueprint\": {
    \"answer_block_focus\": \"<string>\",
    \"comparison_table_idea\": \"<string>\",
    \"faq_questions\": [\"<question>\", \"<question>\"]
  },
  \"suggested_cta\": \"<string>\"
}
Rules:
- secondary_keywords: 2-4 items.
- key_talking_points: 3-5 items, each must include at least one E-E-A-T signal (statistic, certification, expert quote, clinical study, etc.).
- faq_questions: exactly 2-4 items.
- All enum fields must use exactly one of the listed values.
- Language of text fields must match the language parameter."""

test_prompt = [
    {"role": "system", "content": system_cb},
    {"role": "user", "content": "Primary keyword: buy sustainable yoga mat\nNiche: Fitness & Supplements\nLanguage: English (US)\nGenerate a content brief."}
]

inputs = tokenizer.apply_chat_template(
    test_prompt,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to("cuda")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.3,
        repetition_penalty=1.08,
        do_sample=True
    )

input_len = inputs["input_ids"].shape[1]
decoded = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)
print("Generated Output:\n", decoded)